<a href="https://colab.research.google.com/github/m4vic/model-finetuning-labs/blob/main/deberta_finetune_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── 1. Install ──────────────────────────────────────────────────────────────────
!pip install -q transformers datasets accelerate sentencepiece protobuf scikit-learn

In [ ]:
# ── 2. Check GPU ────────────────────────────────────────────────────────────────
import torch
print(f"GPU  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — go to Runtime → Change runtime → T4'}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if torch.cuda.is_available() else "")
print(f"Torch: {torch.__version__}")

In [ ]:
# ── 4. Config ───────────────────────────────────────────────────────────────────
MODEL_NAME   = "microsoft/deberta-v3-small"
DATASET_ID   = "neuralchemy/Prompt-injection-dataset"
OUTPUT_REPO  = "neuralchemy/prompt-injection-deberta"

MAX_LEN      = 256
BATCH_SIZE   = 16
GRAD_ACCUM   = 2          # effective batch = 32
EPOCHS       = 5
LR           = 5e-6       # KEY: much lower LR prevents DeBERTa-v3 NaN
WARMUP_RATIO = 0.20       # more warmup = smoother start
WEIGHT_DECAY = 0.01
ADAM_EPS     = 1e-6       # KEY fix from DeBERTa-v3 paper (default 1e-8 causes NaN)
GRAD_CLIP    = 1.0
PATIENCE     = 2
OUTPUT_DIR   = "/content/deberta-checkpoint"

print("Config ready. adam_epsilon=1e-6 is the key stability fix for DeBERTa-v3.")

In [ ]:
# ── 5. Load Dataset ─────────────────────────────────────────────────────────────
from datasets import load_dataset

raw = load_dataset(DATASET_ID, "full")
raw = raw.rename_column("label", "labels")
print(raw)

In [ ]:
# ── 6. Tokenize ─────────────────────────────────────────────────────────────────
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

# Remove all columns except labels
cols_to_remove = [c for c in raw["train"].column_names if c != "labels"]
tokenized = raw.map(tokenize_fn, batched=True, remove_columns=cols_to_remove)
tokenized.set_format("torch")

collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenized:", tokenized)

In [ ]:
# ── 7. NaN-safe Metrics ─────────────────────────────────────────────────────────
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Safety: replace NaN logits to prevent cascade failures
    if np.isnan(logits).any():
        print("WARNING: NaN detected in logits — returning 0 metrics for this eval step")
        return {"accuracy": 0.0, "f1": 0.0, "roc_auc": 0.0}

    preds = np.argmax(logits, axis=-1)

    # Stable softmax for probabilities
    shifted = logits - logits.max(axis=-1, keepdims=True)
    exp_l   = np.exp(shifted)
    probs   = (exp_l / exp_l.sum(axis=-1, keepdims=True))[:, 1]

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="binary", zero_division=0)

    try:
        auc = roc_auc_score(labels, probs)
    except Exception:
        auc = 0.0

    # FPR
    try:
        tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    except Exception:
        fpr = 0.0

    return {
        "accuracy":            round(float(acc), 4),
        "f1":                  round(float(f1),  4),
        "roc_auc":             round(float(auc), 4),
        "false_positive_rate": round(float(fpr), 4),
    }

In [ ]:
# ── 8. Load Model ────────────────────────────────────────────────────────────────
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

In [ ]:
# ── 9. Training Arguments ────────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    max_grad_norm               = GRAD_CLIP,
    adam_epsilon                = ADAM_EPS,      # <-- KEY FIX for DeBERTa-v3 NaN
    fp16                        = False,          # FP32 — stable on T4
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    logging_steps               = 50,
    save_total_limit            = 1,
    report_to                   = "none",
    seed                        = 42,
)

trainer = Trainer(
    model            = model,
    args             = args,
    train_dataset    = tokenized["train"],
    eval_dataset     = tokenized["validation"],
    processing_class = tokenizer,
    data_collator    = collator,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

In [ ]:
# ── 10. Train ────────────────────────────────────────────────────────────────────
# Expected: ~35-45 min on T4 (FP32)
# Watch: Epoch 1 val F1 should be 0.95+. If still NaN, see cell 10b for debug.
trainer.train()

In [ ]:
# ── 10b. Debug Cell (run only if training fails) ──────────────────────────────
# Checks a single batch for NaN values before training
from torch.utils.data import DataLoader

loader = DataLoader(tokenized["train"], batch_size=4, collate_fn=collator)
batch  = next(iter(loader))
batch  = {k: v.to("cuda") for k, v in batch.items()}

model.eval()
with torch.no_grad():
    out = model(**batch)

print("Logits:", out.logits)
print("Loss:  ", out.loss.item())
nan_check = torch.isnan(out.logits).any()
print("NaN in logits:", nan_check.item())
model.train()

In [ ]:
# ── 11. Test Evaluation ──────────────────────────────────────────────────────────
test_results = trainer.evaluate(tokenized["test"])

print("\n" + "="*45)
print("  FINAL TEST RESULTS")
print("="*45)
for k, v in test_results.items():
    print(f"  {k:<30} {v}")

# Baseline comparison
rf_f1, rf_auc = 0.9688, 0.9944
print("="*45)
print(f"  vs RF → F1 delta:  {test_results.get('eval_f1', 0) - rf_f1:+.4f}")
print(f"  vs RF → AUC delta: {test_results.get('eval_roc_auc', 0) - rf_auc:+.4f}")

In [ ]:
# ── 13. Save training log to disk (optional) ──────────────────────────────────────
import json
state = trainer.state
report = {
    "best_metric": state.best_metric,
    "best_model_checkpoint": state.best_model_checkpoint,
    "log_history": state.log_history,
    "test": test_results,
}
with open("/content/deberta_results.json", "w") as f:
    json.dump(report, f, indent=2)

from google.colab import files
files.download("/content/deberta_results.json")